# IRP 调仓助手（日常使用）

这份 notebook 参考你现有的 **RP 调仓助手** 与 **IRP 回测 notebook**，复用已有的：

- `market_data.py`
- `backtest.py`
- `intrinsic_backtest.py`

目标是：

1. 定义资产池  
2. 输入当前持仓（按**手**）、成本价、可用现金  
3. 从本地数据库读取最新市场数据  
4. 计算最新目标 **本征风险平价（IRP）** 权重  
5. 在 **整手、long-only、考虑交易成本、考虑成交额占比限制** 下，给出调仓建议

## 说明

- 这里的建议基于 **数据库中最新可用交易日** 的数据
- 成交价使用你选择的 `execution_price_type`（默认 `avg`）
- 若某资产当日无可交易价格，则不会对其给出成交建议
- 若资金不足，会自然放弃部分标的或仅部分逼近目标权重
- 和 RP 不同的是，IRP 还需要：
  - `lookback_window`（lbw）
  - `long_lookback_window`（llbw）
  - `activity_field`（如 `amount` / `vol`）
- 本 notebook 会自动按 `llbw + 1` 向前多读取一段历史数据，避免因为样本不足拿不到 IRP 权重


In [ ]:

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from market_data import create_manager, today_str, load_tushare_token
from backtest import (
    build_market_matrices,
    get_execution_price_matrix,
    rebalance_to_target_weights,
    calc_actual_weights,
    calc_portfolio_value,
)
from intrinsic_backtest import compute_intrinsic_target_weights_on_date

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True

# ========= 连接数据库 =========
TUSHARE_TOKEN = load_tushare_token()
DB_PATH = "data/db/market_data.db"

manager = create_manager(
    tushare_token=TUSHARE_TOKEN,
    db_path=DB_PATH,
    default_start_date="20150101",
    default_exchange="SSE",
)

print("manager ready")
print("db_path =", Path(DB_PATH).resolve())


In [ ]:

# ========= 输入区：资产池 / 当前持仓 / 成本价 / 现金 / 参数 =========

WATCHLIST = [
    "510300.SH",  # 沪深300ETF
    "511090.SH",  # 30年国债ETF
    "518880.SH",  # 黄金ETF
    "159981.SZ",  # 能源ETF
    "159985.SZ",  # 豆粕ETF
    "501018.SH",  # 南方原油LOF
    "515100.SH",  # 红利低波100ETF
    "159659.SZ",  # 100纳斯达克ETF
]

# 当前持仓：按“手”填写，1 手 = 100 份（默认）
CURRENT_LOTS = {
    "510300.SH": 0,
    "511090.SH": 0,
    "518880.SH": 0,
    "159981.SZ": 0,
    "159985.SZ": 0,
    "501018.SH": 0,
    "515100.SH": 0,
    "159659.SZ": 0,
}

# 成本价：按“每份/每股价格”填写；没有就写 0 或删掉
COST_PRICE = {
    "510300.SH": 0.0,
    "511090.SH": 0.0,
    "518880.SH": 0.0,
    "159981.SZ": 0.0,
    "159985.SZ": 0.0,
    "501018.SH": 0.0,
    "515100.SH": 0.0,
    "159659.SZ": 0.0,
}

AVAILABLE_CASH = 1_000_000.0

# ===== 调仓参数 =====
LOT_SIZE = 100
EXECUTION_PRICE_TYPE = "avg"     # open / close / high / low / avg
FEE_RATE_BUY = 0.0005
FEE_RATE_SELL = 0.0005

# Tushare fund_daily 的 amount 单位通常是“千元”
MAX_TRADE_AMOUNT_RATIO = 0.05
AMOUNT_UNIT_SCALE = 1000.0

# ===== IRP 参数 =====
LOOKBACK_WINDOW = 120           # lbw：本征协方差矩阵风险窗口
LONG_LOOKBACK_WINDOW = 252      # llbw：zr / zv 长期标准化窗口
ACTIVITY_FIELD = "amount"       # amount / vol
VALUATION_FFILL_LIMIT = 5

IRP_PREPARE_KWARGS = {
    "ffill": True,
    "ffill_limit": 5,
    "min_non_na_ratio": 0.8,
    "drop_all_na_dates": True,
}

# 一般不建议对成交量/成交额前向填充，以免制造虚假活跃度
ACTIVITY_PREPARE_KWARGS = {
    "ffill": False,
    "min_non_na_ratio": 0.8,
    "drop_all_na_dates": True,
}

IRP_WEIGHT_KWARGS = {
    "annualization": 252,
    "drop_any_na": True,
    "long_only": True,
    "return_type": "log",
    "tol": 1e-12,
    "maxiter": 10000,
}

# ===== 数据读取参数 =====
REFRESH_LOOKBACK_DAYS = 180
HISTORY_BUFFER_DAYS = 30   # 在 llbw+1 基础上额外多读一点，提升稳定性

print("watchlist =", WATCHLIST)
print("current lots =", CURRENT_LOTS)
print("available cash =", AVAILABLE_CASH)
print("activity_field =", ACTIVITY_FIELD)
print("llbw =", LONG_LOOKBACK_WINDOW)


In [ ]:

# ========= 刷新最近数据 + 读取本地市场数据并构建 market =========
TRADE_UNIVERSE = sorted(set(WATCHLIST) | set(CURRENT_LOTS.keys()) | set(COST_PRICE.keys()))

recent_summary = manager.refresh_recent_daily_prices(
    ts_codes=TRADE_UNIVERSE,
    lookback_days=REFRESH_LOOKBACK_DAYS,
    end_date=today_str(),
)

print("最近数据刷新结果：")
print(recent_summary)

# 计算自动回看起点：至少覆盖 llbw + 1，再多读一些缓冲
history_need = LONG_LOOKBACK_WINDOW + 1 + HISTORY_BUFFER_DAYS

cal_df = manager.store.get_trade_calendar(
    exchange="SSE",
    start_date="20100101",
    end_date=today_str(),
    is_open=1,
)

open_dates = pd.to_datetime(cal_df["cal_date"], format="%Y%m%d", errors="coerce").dropna().sort_values().reset_index(drop=True)
if len(open_dates) == 0:
    raise ValueError("trade_calendar 为空，无法推导历史起点，请先初始化或更新交易日历。")

history_start_pos = max(0, len(open_dates) - history_need)
HISTORY_START_DATE = open_dates.iloc[history_start_pos].strftime("%Y%m%d")
END_DATE = today_str()

fields = ["open", "high", "low", "close", "amount"]
if ACTIVITY_FIELD not in fields:
    fields.append(ACTIVITY_FIELD)

raw_prices = manager.store.get_daily_prices(
    ts_codes=TRADE_UNIVERSE,
    start_date=HISTORY_START_DATE,
    end_date=END_DATE,
)

print("history_start_date =", HISTORY_START_DATE)
print("end_date =", END_DATE)
print("raw_prices shape =", raw_prices.shape)
display(raw_prices.head())

coverage = raw_prices.groupby("ts_code")["trade_date"].agg(["min", "max", "count"])
print("资产覆盖情况：")
display(coverage)

market = build_market_matrices(
    data=raw_prices,
    fields=tuple(fields),
    date_col="trade_date",
    code_col="ts_code",
    date_format="%Y%m%d",
)

close_px = market["close"].copy()
val_px = close_px.ffill(limit=VALUATION_FFILL_LIMIT)
exec_px = get_execution_price_matrix(market, execution_price_type=EXECUTION_PRICE_TYPE)
amount_px = market["amount"].copy()

latest_signal_date = close_px.index.max()
val_today = val_px.loc[latest_signal_date]
exec_today = exec_px.loc[latest_signal_date]
amount_today = amount_px.loc[latest_signal_date]

# 为避免反复运行 notebook 时改写原字典，这里单独构造局部 kwargs
irp_prepare_kwargs = dict(IRP_PREPARE_KWARGS)
irp_prepare_kwargs["calendar"] = market["close"].index

activity_prepare_kwargs = dict(ACTIVITY_PREPARE_KWARGS)
activity_prepare_kwargs["calendar"] = market["close"].index

print("latest signal date =", latest_signal_date)
for k, v in market.items():
    print(k, v.shape)


In [ ]:

# ========= 计算当前组合状态 =========
codes = close_px.columns.tolist()

current_shares = pd.Series(0, index=codes, dtype=int)
for code, lots in CURRENT_LOTS.items():
    if code in current_shares.index:
        current_shares.loc[code] = int(lots) * LOT_SIZE

cost_price_series = pd.Series(0.0, index=codes, dtype=float)
for code, px in COST_PRICE.items():
    if code in cost_price_series.index:
        cost_price_series.loc[code] = float(px)

current_market_value = (current_shares * val_today.reindex(codes)).fillna(0.0)
current_actual_weights = calc_actual_weights(current_shares, val_today, cash=AVAILABLE_CASH).reindex(codes).fillna(0.0)
portfolio_value = calc_portfolio_value(current_shares, val_today, AVAILABLE_CASH)
cash_weight = AVAILABLE_CASH / portfolio_value if portfolio_value > 0 else np.nan

current_summary = pd.DataFrame({
    "lots": current_shares / LOT_SIZE,
    "shares": current_shares,
    "cost_price": cost_price_series,
    "valuation_price": val_today.reindex(codes),
    "execution_price": exec_today.reindex(codes),
    "market_value": current_market_value,
    "actual_weight": current_actual_weights,
})

current_summary["pnl"] = (current_summary["valuation_price"] - current_summary["cost_price"]) * current_summary["shares"]
current_summary["pnl_pct"] = np.where(
    current_summary["cost_price"] > 0,
    current_summary["valuation_price"] / current_summary["cost_price"] - 1.0,
    np.nan,
)

print("current portfolio value =", round(portfolio_value, 2))
print("cash =", round(AVAILABLE_CASH, 2))
print("cash weight =", round(cash_weight, 4))

display(current_summary)


In [ ]:

# ========= 计算目标 IRP 权重 =========
# 只在 WATCHLIST 上计算 IRP 权重
watchlist_market = {k: v.reindex(columns=WATCHLIST) for k, v in market.items()}

target_weights_watchlist = compute_intrinsic_target_weights_on_date(
    market=watchlist_market,
    signal_date=latest_signal_date,
    lookback_window=LOOKBACK_WINDOW,
    long_lookback_window=LONG_LOOKBACK_WINDOW,
    activity_field=ACTIVITY_FIELD,
    irp_prepare_kwargs=irp_prepare_kwargs,
    activity_prepare_kwargs=activity_prepare_kwargs,
    irp_weight_kwargs=IRP_WEIGHT_KWARGS,
).reindex(WATCHLIST).fillna(0.0)

if float(target_weights_watchlist.sum()) <= 0:
    raise ValueError(
        "IRP 目标权重为空。请检查：\n"
        "1) 历史数据是否足够覆盖 llbw+1；\n"
        "2) activity_field 是否存在且有数据；\n"
        "3) 某些资产是否因缺失过多被预处理剔除。"
    )

# 扩展到全部交易范围：不在资产池的当前持仓，目标权重=0
target_weights = pd.Series(0.0, index=codes, dtype=float)
target_weights.loc[target_weights_watchlist.index] = target_weights_watchlist.values

target_weight_df = target_weights.to_frame("target_weight")
display(target_weight_df.sort_values("target_weight", ascending=False))


In [ ]:

# ========= 生成调仓建议 =========
new_shares, new_cash, trades_df, after_weights = rebalance_to_target_weights(
    current_shares=current_shares,
    cash=AVAILABLE_CASH,
    target_weights=target_weights,
    exec_prices=exec_today,
    val_prices=val_today,
    amount_series=amount_today,
    fee_rate_buy=FEE_RATE_BUY,
    fee_rate_sell=FEE_RATE_SELL,
    lot_size=LOT_SIZE,
    max_trade_amount_ratio=MAX_TRADE_AMOUNT_RATIO,
    amount_unit_scale=AMOUNT_UNIT_SCALE,
    trade_date=latest_signal_date,
)

after_market_value = (new_shares * val_today.reindex(codes)).fillna(0.0)
after_weights = after_weights.reindex(codes).fillna(0.0)

if len(trades_df) > 0:
    buy_df = trades_df[trades_df["side"] == "BUY"].groupby("ts_code")["shares"].sum().rename("buy_shares")
    sell_df = trades_df[trades_df["side"] == "SELL"].groupby("ts_code")["shares"].sum().rename("sell_shares")
    trade_value_df = trades_df.groupby("ts_code")["trade_value"].sum().rename("trade_value")
    trade_cost_df = trades_df.groupby("ts_code")["cost"].sum().rename("trade_cost")
else:
    buy_df = pd.Series(dtype=float, name="buy_shares")
    sell_df = pd.Series(dtype=float, name="sell_shares")
    trade_value_df = pd.Series(dtype=float, name="trade_value")
    trade_cost_df = pd.Series(dtype=float, name="trade_cost")

rebalance_table = pd.DataFrame(index=codes)
rebalance_table["current_lots"] = current_shares / LOT_SIZE
rebalance_table["current_shares"] = current_shares
rebalance_table["target_weight"] = target_weights
rebalance_table["current_weight"] = current_actual_weights
rebalance_table["after_weight"] = after_weights
rebalance_table["valuation_price"] = val_today.reindex(codes)
rebalance_table["execution_price"] = exec_today.reindex(codes)
rebalance_table = rebalance_table.join(buy_df, how="left").join(sell_df, how="left")
rebalance_table = rebalance_table.join(trade_value_df, how="left").join(trade_cost_df, how="left")
rebalance_table[["buy_shares", "sell_shares", "trade_value", "trade_cost"]] = rebalance_table[
    ["buy_shares", "sell_shares", "trade_value", "trade_cost"]
].fillna(0.0)

rebalance_table["net_trade_shares"] = rebalance_table["buy_shares"] - rebalance_table["sell_shares"]
rebalance_table["net_trade_lots"] = rebalance_table["net_trade_shares"] / LOT_SIZE
rebalance_table["after_shares"] = new_shares.reindex(codes).fillna(0).astype(int)
rebalance_table["after_lots"] = rebalance_table["after_shares"] / LOT_SIZE
rebalance_table["after_market_value"] = after_market_value

def classify_action(x):
    if x > 0:
        return "BUY"
    if x < 0:
        return "SELL"
    return "HOLD"

rebalance_table["suggestion"] = rebalance_table["net_trade_shares"].apply(classify_action)

print("cash before =", round(AVAILABLE_CASH, 2))
print("cash after  =", round(new_cash, 2))
print("cash delta  =", round(new_cash - AVAILABLE_CASH, 2))
print("trade count =", len(trades_df))

display(rebalance_table.sort_values(["suggestion", "target_weight"], ascending=[True, False]))


In [ ]:

# ========= 仅看需要操作的标的 =========
action_table = rebalance_table[rebalance_table["suggestion"] != "HOLD"].copy()
display(action_table.sort_values("net_trade_shares", ascending=False))

print("交易明细：")
display(trades_df)


In [ ]:

# ========= 图形查看：当前权重 / 目标权重 / 调后权重 =========
compare_weights = pd.DataFrame({
    "current_weight": current_actual_weights,
    "target_weight": target_weights,
    "after_weight": after_weights,
}).fillna(0.0)

display(compare_weights.sort_values("target_weight", ascending=False))

compare_weights.plot(kind="bar", figsize=(14, 5))
plt.title("Current vs Target vs After-Rebalance Weights (IRP)")
plt.ylabel("Weight")
plt.tight_layout()
plt.show()


In [ ]:

# ========= 导出建议 =========
EXPORT_DIR = Path("data/exports_intrinsic_rebalance")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

current_summary.to_csv(EXPORT_DIR / "irp_rebalance_current_summary.csv")
target_weight_df.to_csv(EXPORT_DIR / "irp_rebalance_target_weights.csv")
rebalance_table.to_csv(EXPORT_DIR / "irp_rebalance_suggestion.csv")
trades_df.to_csv(EXPORT_DIR / "irp_rebalance_trades_detail.csv", index=False)

print("export done")
print("export_dir =", EXPORT_DIR.resolve())


In [ ]:

# ========= 精简版调仓建议（含标的名） =========
signal_date = pd.Timestamp(latest_signal_date)
signal_date_str = signal_date.strftime("%Y-%m-%d")

# 读取标的名称映射
inst_df = manager.store.get_instruments(listed_only=False)
name_map = dict(zip(inst_df["ts_code"], inst_df["name"]))

# 尝试从交易日历里找下一个交易日
next_trade_date = None
try:
    cal_df = manager.store.get_trade_calendar(
        exchange="SSE",
        start_date=signal_date.strftime("%Y%m%d"),
        end_date="20300101",
        is_open=1,
    )
    future_open_dates = pd.to_datetime(cal_df["cal_date"], format="%Y%m%d", errors="coerce")
    future_open_dates = future_open_dates[future_open_dates > signal_date]
    if len(future_open_dates) > 0:
        next_trade_date = future_open_dates.iloc[0]
except Exception:
    next_trade_date = None

simple_advice = rebalance_table[["current_lots", "after_lots", "target_weight"]].copy()

# 转成展示格式
simple_advice = simple_advice.reset_index().rename(columns={
    "index": "ts_code",
    "current_lots": "调整前手数",
    "after_lots": "调整后手数",
    "target_weight": "目标权重",
})

# 增加标的名称
simple_advice["name"] = simple_advice["ts_code"].map(name_map)

# 调整列顺序
simple_advice = simple_advice[["ts_code", "name", "调整前手数", "调整后手数", "目标权重"]]

# 手数转整数
simple_advice["调整前手数"] = simple_advice["调整前手数"].astype(int)
simple_advice["调整后手数"] = simple_advice["调整后手数"].astype(int)

print(f"当前读取到的最新市场数据日期：{signal_date_str}")
print(f"IRP 参数：lbw={LOOKBACK_WINDOW}, llbw={LONG_LOOKBACK_WINDOW}, activity={ACTIVITY_FIELD}")

if next_trade_date is not None:
    print(f"建议于下一个交易日（{next_trade_date.strftime('%Y-%m-%d')}）做如下调整：")
else:
    print("建议于下一个交易日做如下调整：")

display(simple_advice)
